# Fashion-MNIST CNN experiment
Run all cells from a fresh kernel after installing requirements.txt. First use requires internet access to download Fashion-MNIST. Training is configured for 100 epochs and can take significant time.
Historical coursework metrics and figures are preserved in the README/report. The updated notebook has cleared outputs; the code has been smoke-tested on a synthetic batch, not retrained to reproduce the historical accuracy.


In [ ]:
import numpy as np
from tensorflow.keras import layers, models
import tensorflow as tf
import matplotlib.pyplot as plt
from data_utils import load_mnist, prepare_images

SEED = 42
EPOCHS = 100
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()


In [ ]:
# Explicit dataset identity; first use downloads Fashion-MNIST through Keras.
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.fashion_mnist.load_data()
# For matching local IDX files, use load_mnist(path, kind="train"/"t10k") instead.


In [ ]:
train_labels = tf.keras.utils.to_categorical(train_labels, 10)
test_labels = tf.keras.utils.to_categorical(test_labels, 10)

In [ ]:
train_images = prepare_images(train_images)
test_images = prepare_images(test_images)


In [ ]:
from sklearn.model_selection import train_test_split

# Splitting training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(train_images, train_labels, test_size=0.2, random_state=SEED)

## Training policy
This experiment trains on normalized images without augmentation, matching the original executed training path. The previously unused augmentation generator was removed. Set `EPOCHS` above for a shorter exploratory run.


In [ ]:
model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9), 
              loss='categorical_crossentropy',  
              metrics=['accuracy'])

history = model.fit(
    X_train, y_train, 
    epochs=EPOCHS,
    batch_size=128,
    validation_data=(X_val, y_val)
)

In [ ]:
# Plot for accuracy
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)  # 1 row, 2 columns, first plot
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Plot for loss
plt.subplot(1, 2, 2)  # 1 row, 2 columns, second plot
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.show()

In [ ]:
test_loss, test_accuracy = model.evaluate(test_images, test_labels)
print(f'Test loss: {test_loss}, Test accuracy: {test_accuracy}')
model.save('fashion_mnist_model.keras')